In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import json
from datetime import datetime
import logging

### Пример класса модели прогнозирования `(LSTM)`

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_size=5, hidden_size=50, num_layers=2, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Берем только последний выход последовательности
        last_output = lstm_out[:, -1, :]
        last_output = self.dropout(last_output)
        
        output = self.linear(last_output)
        return output


In [ ]:
class StockDataPreprocessor:
    def __init__(self, sequence_length=10):
        self.sequence_length = sequence_length
        self.scaler = MinMaxScaler()
        self.feature_scaler = MinMaxScaler()
        
    def prepare_data(self, data, features=['open', 'high', 'low', 'close', 'volume']):
        """Подготовка данных для обучения"""
        df = pd.DataFrame(data)
        
        # Извлекаем фичи и целевую переменную
        feature_data = df[features].values
        target_data = df['close'].values.reshape(-1, 1)
        
        # Нормализуем данные
        feature_scaled = self.feature_scaler.fit_transform(feature_data)
        target_scaled = self.scaler.fit_transform(target_data)
        
        return self.create_sequences(feature_scaled, target_scaled)
    
    def create_sequences(self, features, targets):
        """Создание последовательностей для LSTM"""
        X, y = [], []
        
        for i in range(len(features) - self.sequence_length):
            X.append(features[i:(i + self.sequence_length)])
            y.append(targets[i + self.sequence_length])
            
        return np.array(X), np.array(y)

In [ ]:
def generate_sample_data(num_points=1000):
    """Генерация синтетических данных акций для примера"""
    data = []
    price = 100.0
    
    for i in range(num_points):
        change = np.random.normal(0, 1.5)
        volume = np.random.randint(1000000, 50000000)
        
        open_price = price
        close_price = price + change
        high_price = max(open_price, close_price) + abs(np.random.normal(0, 0.5))
        low_price = min(open_price, close_price) - abs(np.random.normal(0, 0.5))
        
        data.append({
            'datetime': f'2024-01-{(i//78)+1:02d} {(i%78)//60:02d}:{(i%78)%60:02d}:00',
            'open': float(open_price),
            'high': float(high_price),
            'low': float(low_price),
            'close': float(close_price),
            'volume': volume,
            'value': float(volume * close_price)
        })
        
        price = close_price
    
    return data

In [8]:
def train_model():
    """Обучение LSTM модели"""
    logging.basicConfig(level=logging.INFO)
    
    # Генерируем или загружаем данные
    print("Генерация данных")
    train_data = generate_sample_data(800)
    test_data = generate_sample_data(200)
    
    # Подготавливаем данные
    preprocessor = StockDataPreprocessor(sequence_length=10)
    X_train, y_train = preprocessor.prepare_data(train_data)
    X_test, y_test = preprocessor.prepare_data(test_data)
    
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train)
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test)
    
    print(f"Размер train: {X_train.shape}")
    print(f"Размер test: {X_test.shape}")
    
    # Создаем модель
    model = LSTMPredictor(
        input_size=5,
        hidden_size=50,
        num_layers=2,
        output_size=1,
        dropout=0.2
    )
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    # Обучение
    num_epochs = 50
    train_losses = []
    val_losses = []
    
    print("Начало обучения")
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Валидация
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_test)
            val_loss = criterion(val_outputs, y_test)
            
        scheduler.step(val_loss)
        
        train_losses.append(loss.item())
        val_losses.append(val_loss.item())
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.6f}, Val Loss: {val_loss.item():.6f}')
    
    # Сохраняем модель разными способами (можно выбрать один из них для дальнейшей отправки модели)
    print("Сохранение модели...")
    
    # 1. Сохраняем веса модели
    torch.save(model.state_dict(), 'lstm_model_weights.pth')
    
    # 2. Сохраняем полную модель (архитектура + веса)
    torch.save(model, 'lstm_model_complete.pth')
    
    # 3. Сохраняем как TorchScript
    model.eval()
    example_input = torch.randn(1, 10, 5)  # batch_size=1, seq_len=10, features=5
    traced_model = torch.jit.trace(model, example_input)
    traced_model.save('lstm_model_scripted.pt')
    
    # Сохраняем препроцессор
    preprocessing_info = {
        'sequence_length': preprocessor.sequence_length,
        'feature_names': ['open', 'high', 'low', 'close', 'volume'],
        'scaler_min': preprocessor.scaler.min_.tolist(),
        'scaler_scale': preprocessor.scaler.scale_.tolist(),
        'feature_scaler_min': preprocessor.feature_scaler.min_.tolist(),
        'feature_scaler_scale': preprocessor.feature_scaler.scale_.tolist()
    }
    
    with open('preprocessing_info.json', 'w') as f:
        json.dump(preprocessing_info, f, indent=2)
    
    # Сохраняем код модели отдельно
    model_code = '''
import torch
import torch.nn as nn

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=5, hidden_size=50, num_layers=2, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        last_output = self.dropout(last_output)
        output = self.linear(last_output)
        return output
'''
    
    with open('model_architecture.py', 'w') as f:
        f.write(model_code)
    
    # Сохраняем тестовые данные - на них будут считаться метрики
    test_dataset = {
        'dataset': test_data[:100]  # Берем первые 100 точек для теста
    }
    
    with open('test_data.json', 'w') as f:
        json.dump(test_dataset, f, indent=2)
    
    print("Обучение завершено! Сохранены:")
    print("- lstm_model_weights.pth (только веса)")
    print("- lstm_model_complete.pth (полная модель)")
    print("- lstm_model_scripted.pt (torchscript)")
    print("- model_architecture.py (код модели)")
    print("- preprocessing_info.json (параметры препроцессинга)")
    print("- test_data.json (тестовые данные)")

In [9]:
train_model()

Генерация данных
Размер train: torch.Size([790, 10, 5])
Размер test: torch.Size([190, 10, 5])
Начало обучения
Epoch [10/50], Train Loss: 0.138228, Val Loss: 0.149365
Epoch [20/50], Train Loss: 0.061545, Val Loss: 0.061716
Epoch [30/50], Train Loss: 0.029704, Val Loss: 0.043006
Epoch [40/50], Train Loss: 0.028776, Val Loss: 0.043441
Epoch [50/50], Train Loss: 0.027465, Val Loss: 0.040987
Сохранение модели...
Обучение завершено! Сохранены:
- lstm_model_weights.pth (только веса)
- lstm_model_complete.pth (полная модель)
- lstm_model_scripted.pt (torchscript)
- model_architecture.py (код модели)
- preprocessing_info.json (параметры препроцессинга)
- test_data.json (тестовые данные)
